In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import random
import time

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_procs as sqlp
import ferdinand.download_procs as dlp

from pathlib import Path
from types import SimpleNamespace


# Purpose and Documentation

## Purpose 
Download current radiograph image files from the IMPC image server based on the metadata CSV generated by <code style="font-size:12px;">010_fetch_data_from_impc.ipynb</code>.

## Location 

<code style="font-size:12px;">&lt;project_dir&gt;/notebooks/020_download_impc_radiographs.ipynb</code>

## Workflow

1. <b>Clean up</b>: Remove database records with ambiguous OMERO IDs
2. <b>Download</b>: Fetch image files (both DICOM and JPEG versions) from IMPC servers
3. <b>Track metadata</b>: Save image metadata (dimensions, file types, spacing) to database
4. <b>Export</b>: Write updated CSV with download status back to <code style="font-size:12px;">IMPC_radiographs.csv</code>

## Prerequisites
- <b>Configuration</b>: <code style="font-size:12px;">../config/config.yml</code> with <code style="font-size:12px;">centers</code>, <code style="font-size:12px;">run_name</code>, and <code style="font-size:12px;">directories</code> properly configured
- <b>Input CSV</b>: <code style="font-size:12px;">../data/&lt;run_name&gt;/030_all_data/IMPC_radiographs.csv</code> (output from notebook 010_fetch_data_from_impc.ipynb)
- <b>Previous notebook execution</b>: Template cells from NNN_notebook_template.ipynb (setup imports, logging, config, and paths)
- <b>Directories</b>: <code style="font-size:12px;">../data/</code> and <code style="font-size:12px;">../reports/</code> directories exist (subdirectories created automatically)
- <b>Network access</b>: Connection to IMPC image servers (https://www.mousephenotype.org)

# Definition / Settings

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_file = list(Path.cwd().glob('*.ipynb'))[0].stem 

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_file}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, config['sqlite_db'])

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlp.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

# Clean up entries with ambiguous OMERO identifiers

In [ ]:
# remove database records that have ambiguous OMERO IDs (e.g. same ID linked to multiple mouse IDs)
sqlp.remove_files_with_ambiguous_omero_ids(conn, db_table='impc_data', logger=logger)

# Download image files and update database records

In [ ]:
# download settings
# mode: REAL=download everything, TEST=only 20 rows
mode = "TEST"

## Get current download status

In [ ]:
total_number = len(sqlp.select_rows(conn, db_table='impc_data'))
downloaded_df = sqlp.select_rows_by_column(conn, db_table='impc_data', column='downloaded', value='yes', logger=logger)
downloaded_number = len(downloaded_df)
not_yet_downloaded_number = len(sqlp.select_rows_by_column(conn, db_table='impc_data', column='downloaded', value='no', logger=logger))

print(f"             total number of images: {total_number}")
print(f"        number of downloaded images: {downloaded_number}")
print(f"number of not yet downloaded images: {not_yet_downloaded_number}")

## Check whether downloaded images exist

In [ ]:
if downloaded_number > 0:
    try: 
        for idx, row in downloaded_df.iterrows():
            local_img_path = utl.build_local_image_dir(dirs.images_dir, row.center, row['cohort_type'], row['gene_symbol'], row['sex'])
            local_img_file = os.path.join(local_img_path, f"{row['omero_id']}.{row['file_extension']}")
            if not os.path.exists(local_img_file): 
                print(f" missing file for omero ID {row['omero_id']} at center {row.center}")
    except NameError as e:
        print(f":-( Error occurred: {e}")

## Fetch the next download batch

In [ ]:
to_download = sqlp.select_next_download_batch(conn, db_table='impc_data', limit=200000)

print(f" Number of files to download:\n  {len(to_download)}")
print(f" Available file types:\n  {to_download.impc_file_type.unique()}")

In [ ]:
logger.setLevel(logging.DEBUG)
print("preparing download ...")

if mode == 'TEST':
    print(f" Number of files to download:\n {len(to_download)}")

for idx, row in to_download.iterrows():

    # random wait time - be nice to the image server
    wait_time = random.uniform(0.1, 2)
    time.sleep(wait_time)
    
    # handle genotype
    gene = f"{row['gene_symbol'].replace(' ', '_')}/" if row['cohort_type']=='mutant' else ''

    # build center/genotype/gene/sex-specific path ..
    local_image_dir = utl.build_local_image_dir(dirs.images_dir, 
                                               row['center'].replace(' ', '_'),
                                               row['cohort_type'], 
                                               gene, 
                                               row['sex'])            
    
    # ... and make sure path exists
    os.makedirs(local_image_dir, exist_ok=True)

    try: 
        print(f"Downloading file {idx} with omero id {row['omero_id']}")
        _df = dlp.download_image(row['omero_id'], local_image_dir, False, True, logger)
        _jpeg_df = dlp.download_image(row['omero_id'], local_image_dir, True, True, logger)
        info_df = pd.merge(left=_df, right=_jpeg_df, on='omero_id', how='outer')
        sqlp.save_full_image_metadata(info_df, conn, 'impc_data') 
        sqlp.update_column_values(conn, db_table='impc_data', column='downloaded', value='yes', condition_column='omero_id', condition_value=row['omero_id'])
    except RuntimeError: 
        sqlp.update_column_values(conn, db_table='impc_data', column='downloaded', value='failed', condition_column='omero_id', condition_value=row['omero_id'])
        print(f"ERROR: Failed to download {local_image_dir}!")

# Update CSV file

In [ ]:
# update IMPC_radiographs.csv
updated_df = pd.read_sql_query("SELECT * from impc_data", conn)
conn.close()

data_file = os.path.join(dirs.all_data_dir, 'IMPC_radiographs.csv')
updated_df.to_csv(data_file, index=False)
print(f" CSV file {data_file} updated")